In [1]:
from langchain.chat_models import ChatOpenAI
# 구체적으로 대답할 필요가 있는 모델에 예제를 준다.
from langchain.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate, ChatPromptTemplate
# 예제 수가 너무 많을 때, 설정(유저의 로그인 여부, 유저가 사용하는 언어 등)에 따라 prompt에 맞는 예제를 필터링 해준다.
from langchain.prompts.example_selector import LengthBasedExampleSelector
# for making your own ExampleSelector
from langchain.prompts.example_selector.base import BaseExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler(),
    ],
)

In [2]:
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

In [3]:
# using FewShotPromptTemplate and PromptTemplate

'''
# use same variables with examples
example_template = """
    Human: {question}\n
    AI: {answer}
"""

# how to format examples
example_prompt = PromptTemplate.from_template(example_template)
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    # question of user
    suffix="Human: What do you know about {country}?",
    # 유효성 검사
    input_variables=["country"],
)

chain = prompt | chat
chain.invoke({"country": "Greenland"})
'''


'\n# use same variables with examples\nexample_template = """\n    Human: {question}\n\n    AI: {answer}\n"""\n\n# how to format examples\nexample_prompt = PromptTemplate.from_template(example_template)\nprompt = FewShotPromptTemplate(\n    example_prompt=example_prompt,\n    examples=examples,\n    # question of user\n    suffix="Human: What do you know about {country}?",\n    # 유효성 검사\n    input_variables=["country"],\n)\n\nchain = prompt | chat\nchain.invoke({"country": "Greenland"})\n'

In [4]:
# using LengthBasedExampleSelector

'''
example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 아래 조건 보다 낮은 길이의 예제만 필터링
    max_length=80,
)

# 아래 FewShotPromptTemplate이 모든 예제에 형식을 지정하기 전에, 위에 먼저 example_selector를 만든다.
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")
'''

'\nexample_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")\n\nexample_selector = LengthBasedExampleSelector(\n    examples=examples,\n    example_prompt=example_prompt,\n    # 아래 조건 보다 낮은 길이의 예제만 필터링\n    max_length=80,\n)\n\n# 아래 FewShotPromptTemplate이 모든 예제에 형식을 지정하기 전에, 위에 먼저 example_selector를 만든다.\nprompt = FewShotPromptTemplate(\n    example_prompt=example_prompt,\n    example_selector=example_selector,\n    suffix="Human: What do you know about {country}?",\n    input_variables=["country"],\n)\n\nprompt.format(country="Brazil")\n'

In [7]:
# making your own ExampleSelector

# 무작위로 하나의 예제만 선택
class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples

    # example list에 example 추가
    def add_example(self, example):
        self.examples.append(example)

    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)]
    
example_prompt = PromptTemplate.from_template("Human: {question}\nAI: {answer}")

example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=["country"],
)

prompt.format(country="Brazil")

'Human: What do you know about Greece?\nAI: \n        I know this:\n        Capital: Athens\n        Language: Greek\n        Food: Souvlaki and Feta Cheese\n        Currency: Euro\n        \n\nHuman: What do you know about Brazil?'

In [ ]:
# using FewShotChatMessagePromptTemplate and ChatPromptTemplate

'''
examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        # correspond to examples
        ("human", "What do you know about {country}?"),
        ("ai", "{answer}"),
    ]
)

#예제의 형식을 example_prompt와 같이 지정
prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    prompt,
    ("human", "What do you know about {country}?")
])

chain = final_prompt | chat
chain.invoke({"country": "Russia"})'
'''